In [ ]:
from datetime import datetime

job_id = session.sql("SELECT UUID_STRING()").collect()[0][0]
job_name = "DWH_FACT_PERSON_ORG_INVOLVEMENT_LOAD"
layer_name = "DWH"
start_time = datetime.now()

rows_loaded = 0
rows_failed = 0
run_status = "SUCCESS"
error_message = None

try:

    fact_poi_df = session.sql(f"""
        SELECT
            poi.POI_ID,
            poi.IC_IC_ID,
            poi.CAS_CAS_ID,
            poi.INV_INV_ID,
            poi.ORGN_ORGN_ID,
            poi.PERSON_PERSON_ID,
            poi.START_DATE,
            poi.END_DATE,
            d.PERSON_ORG_INVOLVEMENT_INFO_ID,
            poi.ADD_TS,
            poi.ADD_USER,
            poi.MOD_TS,
            poi.MOD_USER,
            CURRENT_TIMESTAMP() AS CREATED_DATE,
            poi.DELETED_DATE,
            poi.CHECKSUM,
            poi.SEARCHED_IND
        FROM {STG}.{STG_PERSON_ORG_INVOLVEMENT} poi
        LEFT JOIN {DWH}.{DIM_PERSON_ORG_INVOLVEMENT_INFO} d
            ON poi.REASON_CLOSED_CODE IS NOT DISTINCT FROM d.REASON_CLOSED_CODE_AV_ID
           AND d.UPDATED_DATE IS NULL
    """)

    fact_poi_df.create_or_replace_temp_view("TEMP_FACT_PERSON_ORG_INVOLVEMENT")

    merge_result = session.sql(f"""
        MERGE INTO {DWH}.{FACT_PERSON_ORG_INVOLVEMENT} tgt
        USING TEMP_FACT_PERSON_ORG_INVOLVEMENT src
            ON tgt.POI_ID = src.POI_ID
        WHEN MATCHED AND (
            src.CHECKSUM <> tgt.CHECKSUM
        ) THEN
            UPDATE SET
                IC_IC_ID = src.IC_IC_ID,
                CAS_CAS_ID = src.CAS_CAS_ID,
                INV_INV_ID = src.INV_INV_ID,
                ORGN_ORGN_ID = src.ORGN_ORGN_ID,
                PERSON_PERSON_ID = src.PERSON_PERSON_ID,
                START_DATE = src.START_DATE,
                END_DATE = src.END_DATE,
                PERSON_ORG_INVOLVEMENT_INFO_ID = src.PERSON_ORG_INVOLVEMENT_INFO_ID,
                ADD_TS = src.ADD_TS,
                ADD_USER = src.ADD_USER,
                MOD_TS = src.MOD_TS,
                MOD_USER = src.MOD_USER,
                DELETED_DATE = src.DELETED_DATE,
                CHECKSUM = src.CHECKSUM,
                SEARCHED_IND = src.SEARCHED_IND
        WHEN NOT MATCHED THEN
            INSERT (
                POI_ID,
                IC_IC_ID,
                CAS_CAS_ID,
                INV_INV_ID,
                ORGN_ORGN_ID,
                PERSON_PERSON_ID,
                START_DATE,
                END_DATE,
                PERSON_ORG_INVOLVEMENT_INFO_ID,
                ADD_TS,
                ADD_USER,
                MOD_TS,
                MOD_USER,
                CREATED_DATE,
                DELETED_DATE,
                CHECKSUM,
                SEARCHED_IND
            )
            VALUES (
                src.POI_ID,
                src.IC_IC_ID,
                src.CAS_CAS_ID,
                src.INV_INV_ID,
                src.ORGN_ORGN_ID,
                src.PERSON_PERSON_ID,
                src.START_DATE,
                src.END_DATE,
                src.PERSON_ORG_INVOLVEMENT_INFO_ID,
                src.ADD_TS,
                src.ADD_USER,
                src.MOD_TS,
                src.MOD_USER,
                src.CREATED_DATE,
                src.DELETED_DATE,
                src.CHECKSUM,
                src.SEARCHED_IND
            )
    """).collect()

    rows_inserted = merge_result[0][0]
    rows_updated = merge_result[0][1]
    rows_loaded = rows_inserted + rows_updated

    session.call(
       f"{DB}.{AUDIT}.{SP_LOG_AUDIT}",
        job_id,
        job_name,
        layer_name,
        start_time,
        datetime.now(),
        rows_loaded,
        rows_failed,
        run_status,
        None
    )

    session.call(
       f"{DB}.{UTIL}.{SP_SEND_PIPELINE_NOTIFICATION}",
        run_status,
        job_name,
        layer_name,
        f"FACT_PERSON_ORG_INVOLVEMENT load completed. Rows Inserted: {rows_inserted}, Rows Updated: {rows_updated}"
    )

    print(f"DWH LOAD SUCCESS | Rows Inserted: {rows_inserted} | Rows Updated: {rows_updated}")

except Exception as error:

    run_status = "FAILED"
    rows_failed = 1
    error_message = str(error)

    session.call(
       f"{DB}.{AUDIT}.{SP_LOG_AUDIT}",
        job_id,
        job_name,
        layer_name,
        start_time,
        datetime.now(),
        0,
        rows_failed,
        run_status,
        error_message
    )

    session.call(
       f"{DB}.{UTIL}.{SP_SEND_PIPELINE_NOTIFICATION}",
        run_status,
        job_name,
        layer_name,
        f"FACT_PERSON_ORG_INVOLVEMENT load failed. Error: {error_message}"
    )

    print(f"DWH LOAD FAILED: {error_message}")